In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
from pyspark.sql.types import *

STORAGE = "stecomlakedev"
BRONZE = f"abfss://bronze@{STORAGE}.dfs.core.windows.net/clickstream"
SILVER = f"abfss://silver@{STORAGE}.dfs.core.windows.net/clickstream_events"
QUAR = f"abfss://quarantine@{STORAGE}.dfs.core.windows.net/clickstream_events"
CKPT = f"abfss://checkpoints@{STORAGE}.dfs.core.windows.net/silver_clickstream"
CKPT_QUAR = f"abfss://checkpoints@{STORAGE}.dfs.core.windows.net/quarantine_clickstream"

event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("session_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("page", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("event_ts", StringType(), True),
    StructField("device", StringType(), True),
    # Declared explicitly so the upstream drift lands in a typed column instead
    # of being silently discarded.
    StructField("campaign_id", StringType(), True),
])

VALID_TYPES = ["page_view", "product_view", "add_to_cart",
               "remove_from_cart", "begin_checkout", "purchase"]

# COMMAND ----------
bronze_stream = spark.readStream.format("delta").load(BRONZE)

parsed = (bronze_stream
    .withColumn("j", F.from_json("raw_json", event_schema))
    .select("j.*", "ingest_ts")
    .withColumn("event_ts", F.to_timestamp("event_ts")))

quality = (
    F.col("event_id").isNotNull() &
    F.col("user_id").isNotNull() &
    F.col("event_ts").isNotNull() &
    F.col("event_type").isin(VALID_TYPES) &
    (F.col("price") >= 0)
)

good = parsed.filter(quality)

bad = (parsed.filter(~quality)
    .withColumn("quarantine_reason",
        F.when(F.col("event_id").isNull(), "null_event_id")
         .when(F.col("user_id").isNull(), "null_user_id")
         .when(F.col("event_ts").isNull(), "null_event_ts")
         .when(~F.col("event_type").isin(VALID_TYPES), "invalid_event_type")
         .otherwise("negative_price"))
    .withColumn("quarantined_at", F.current_timestamp()))

# The watermark bounds how long state is kept for deduplication: resends arriving
# more than 2 hours after the event time are not caught.
deduped = (good
    .withWatermark("event_ts", "2 hours")
    .dropDuplicates(["event_id"]))

# COMMAND ----------
q1 = (deduped
    .withColumn("event_date", F.to_date("event_ts"))
    .writeStream.format("delta")
    .option("checkpointLocation", CKPT)
    .partitionBy("event_date")
    .trigger(availableNow=True)
    .start(SILVER))

q2 = (bad.writeStream.format("delta")
    .option("checkpointLocation", CKPT_QUAR)
    .trigger(availableNow=True)
    .start(QUAR))

q1.awaitTermination()
q2.awaitTermination()

# COMMAND ----------
b = spark.read.format("delta").load(BRONZE).count()
s = spark.read.format("delta").load(SILVER).count()
q = spark.read.format("delta").load(QUAR).count()

print(f"bronze     {b}")
print(f"silver     {s}")
print(f"quarantine {q}")
print(f"deduplicated {b - s - q}")

assert spark.read.format("delta").load(SILVER).filter("user_id IS NULL").count() == 0

# COMMAND ----------
display(spark.read.format("delta").load(QUAR)
        .groupBy("quarantine_reason").count().orderBy(F.col("count").desc()))